# AI Investment & Business Impact Analysis

## Objective
Analyze AI adoption and investment across banks, countries, regions, technologies, and use cases.

### Key analysis areas
- Data loading and validation
- Data cleaning and preprocessing
- Descriptive statistics using Pandas and NumPy
- AI spending and cost savings analysis
- ROI and automation analysis
- Country, region, bank, technology and use-case comparisons
- Year-wise trends
- Correlation analysis
- Business insights

**Libraries used:** Pandas, NumPy, Matplotlib, Seaborn

In [1]:
# Import required libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Plot style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
# ============================================
# Load Dataset
# ============================================

import os
from pathlib import Path
import pandas as pd
import numpy as np

# Current working directory
current_dir = Path.cwd()

print("Current Working Directory:")
print(current_dir)

# Search for the CSV file recursively
csv_files = list(current_dir.rglob("AI_Automation_Spend_Raw_Data.csv"))

# If CSV is not found
if not csv_files:
    raise FileNotFoundError(
        "\nAI_Automation_Spend_Raw_Data.csv not found.\n"
        "Make sure the CSV file is inside the project folder."
    )

# Select first matching CSV
FILE_PATH = csv_files[0]

print("\nCSV file found at:")
print(FILE_PATH)

# Load dataset
df = pd.read_csv(FILE_PATH)

print("\nDataset loaded successfully!")
print("Shape:", df.shape)

# Display first 5 records
df.head()

Current Working Directory:
d:\AIAutomation Spend in Banking Sector Analysis


FileNotFoundError: 
AI_Automation_Spend_Raw_Data.csv not found.
Make sure the CSV file is inside the project folder.

## 1. Initial Data Understanding

In [ ]:
# Basic information
print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 records:")
display(df.head())

print("\nLast 5 records:")
display(df.tail())

In [ ]:
# Missing values and duplicate records
missing = df.isnull().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print("Missing values:")
display(missing.to_frame("Missing_Count"))

print("Total duplicate rows:", duplicates)

In [ ]:
# Descriptive statistics
print("Numerical summary:")
display(df.describe().T)

print("\nCategorical summary:")
display(df.describe(include="object").T)

## 2. Data Cleaning and Preprocessing

The following steps:
1. Remove duplicate records.
2. Clean column names.
3. Remove unnecessary spaces from text fields.
4. Convert numeric columns to numeric data types.
5. Validate missing values and invalid numeric values.

In [ ]:
# Clean column names
df.columns = (
    df.columns
      .str.strip()
      .str.replace(" ", "_")
      .str.replace(r"[^A-Za-z0-9_]", "", regex=True)
)

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates().copy()
after = len(df)

# Clean text columns
text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

# Convert numeric columns
numeric_columns = [
    "Year",
    "AI_Spend_USD_Million",
    "Estimated_Cost_Savings_USD_Million",
    "ROI_Multiple",
    "Automation_Rate_Percent",
    "FTE_Hours_Saved_Monthly",
    "AI_Projects"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Duplicate rows removed: {before - after}")
print("Cleaned shape:", df.shape)
display(df.head())

In [ ]:
# Post-cleaning validation
print("Missing values after cleaning:")
display(df.isnull().sum().to_frame("Missing_Count"))

print("Duplicate rows after cleaning:", df.duplicated().sum())

print("\nNumeric columns:")
display(df[numeric_columns].describe().T)

## 3. Feature Engineering

In [ ]:
# Derived business metrics
df["Net_Benefit_USD_Million"] = (
    df["Estimated_Cost_Savings_USD_Million"] - df["AI_Spend_USD_Million"]
)

df["Savings_to_Spend_Ratio"] = np.where(
    df["AI_Spend_USD_Million"] != 0,
    df["Estimated_Cost_Savings_USD_Million"] / df["AI_Spend_USD_Million"],
    np.nan
)

df["Total_Impact_Score"] = (
    df["ROI_Multiple"] *
    (1 + df["Automation_Rate_Percent"] / 100) *
    (1 + df["AI_Projects"] / 10)
)

display(df[[
    "AI_Spend_USD_Million",
    "Estimated_Cost_Savings_USD_Million",
    "Net_Benefit_USD_Million",
    "Savings_to_Spend_Ratio",
    "ROI_Multiple",
    "Automation_Rate_Percent",
    "AI_Projects",
    "Total_Impact_Score"
]].head())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of AI spending
plt.figure(figsize=(10, 6))
sns.histplot(df["AI_Spend_USD_Million"], bins=15, kde=True)
plt.title("Distribution of AI Spending")
plt.xlabel("AI Spend (USD Million)")
plt.ylabel("Number of Records")
plt.tight_layout()
plt.show()

In [ ]:
# AI spending by year
year_spend = df.groupby("Year", as_index=False)["AI_Spend_USD_Million"].sum()

plt.figure(figsize=(10, 6))
sns.lineplot(data=year_spend, x="Year", y="AI_Spend_USD_Million", marker="o")
plt.title("Year-wise AI Spending")
plt.xlabel("Year")
plt.ylabel("AI Spend (USD Million)")
plt.tight_layout()
plt.show()

display(year_spend)

In [ ]:
# Cost savings by year
year_savings = df.groupby("Year", as_index=False)["Estimated_Cost_Savings_USD_Million"].sum()

plt.figure(figsize=(10, 6))
sns.lineplot(data=year_savings, x="Year", y="Estimated_Cost_Savings_USD_Million", marker="o")
plt.title("Year-wise Estimated Cost Savings")
plt.xlabel("Year")
plt.ylabel("Cost Savings (USD Million)")
plt.tight_layout()
plt.show()

In [ ]:
# AI spending by region
region_spend = (
    df.groupby("Region", as_index=False)["AI_Spend_USD_Million"]
      .sum()
      .sort_values("AI_Spend_USD_Million", ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=region_spend, x="AI_Spend_USD_Million", y="Region")
plt.title("AI Spending by Region")
plt.xlabel("AI Spend (USD Million)")
plt.ylabel("Region")
plt.tight_layout()
plt.show()

display(region_spend)

In [ ]:
# AI spending by country
country_spend = (
    df.groupby("Country", as_index=False)["AI_Spend_USD_Million"]
      .sum()
      .sort_values("AI_Spend_USD_Million", ascending=False)
)

plt.figure(figsize=(10, 7))
sns.barplot(data=country_spend, x="AI_Spend_USD_Million", y="Country")
plt.title("AI Spending by Country")
plt.xlabel("AI Spend (USD Million)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

display(country_spend)

In [ ]:
# AI spending by technology
technology_spend = (
    df.groupby("Technology", as_index=False)["AI_Spend_USD_Million"]
      .sum()
      .sort_values("AI_Spend_USD_Million", ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=technology_spend, x="AI_Spend_USD_Million", y="Technology")
plt.title("AI Spending by Technology")
plt.xlabel("AI Spend (USD Million)")
plt.ylabel("Technology")
plt.tight_layout()
plt.show()

In [ ]:
# AI projects by use case
use_case_projects = (
    df.groupby("AI_Use_Case", as_index=False)["AI_Projects"]
      .sum()
      .sort_values("AI_Projects", ascending=False)
)

plt.figure(figsize=(10, 7))
sns.barplot(data=use_case_projects, x="AI_Projects", y="AI_Use_Case")
plt.title("AI Projects by Use Case")
plt.xlabel("Number of AI Projects")
plt.ylabel("AI Use Case")
plt.tight_layout()
plt.show()

In [ ]:
# Average ROI by technology
technology_roi = (
    df.groupby("Technology", as_index=False)["ROI_Multiple"]
      .mean()
      .sort_values("ROI_Multiple", ascending=False)
)

plt.figure(figsize=(10, 6))
sns.barplot(data=technology_roi, x="ROI_Multiple", y="Technology")
plt.title("Average ROI by Technology")
plt.xlabel("Average ROI Multiple")
plt.ylabel("Technology")
plt.tight_layout()
plt.show()

In [ ]:
# Average automation rate by maturity
maturity_automation = (
    df.groupby("Maturity", as_index=False)["Automation_Rate_Percent"]
      .mean()
      .sort_values("Automation_Rate_Percent", ascending=False)
)

plt.figure(figsize=(9, 5))
sns.barplot(data=maturity_automation, x="Maturity", y="Automation_Rate_Percent")
plt.title("Average Automation Rate by AI Maturity")
plt.xlabel("Maturity")
plt.ylabel("Automation Rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Top banks by AI spending
bank_spend = (
    df.groupby("Bank", as_index=False)["AI_Spend_USD_Million"]
      .sum()
      .sort_values("AI_Spend_USD_Million", ascending=False)
)

top_banks = bank_spend.head(10)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_banks, x="AI_Spend_USD_Million", y="Bank")
plt.title("Top Banks by AI Spending")
plt.xlabel("AI Spend (USD Million)")
plt.ylabel("Bank")
plt.tight_layout()
plt.show()

display(top_banks)

## 5. Relationship and Correlation Analysis

In [ ]:
# Correlation matrix
corr_columns = [
    "AI_Spend_USD_Million",
    "Estimated_Cost_Savings_USD_Million",
    "ROI_Multiple",
    "Automation_Rate_Percent",
    "FTE_Hours_Saved_Monthly",
    "AI_Projects",
    "Net_Benefit_USD_Million",
    "Savings_to_Spend_Ratio"
]

correlation = df[corr_columns].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of AI Investment and Impact Metrics")
plt.tight_layout()
plt.show()

In [ ]:
# AI spending vs estimated cost savings
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="AI_Spend_USD_Million",
    y="Estimated_Cost_Savings_USD_Million",
    hue="Region",
    size="AI_Projects",
    sizes=(40, 250),
    alpha=0.8
)
plt.title("AI Spending vs Estimated Cost Savings")
plt.xlabel("AI Spend (USD Million)")
plt.ylabel("Estimated Cost Savings (USD Million)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Automation rate vs FTE hours saved
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="Automation_Rate_Percent",
    y="FTE_Hours_Saved_Monthly",
    hue="Maturity",
    style="Maturity",
    s=90
)
plt.title("Automation Rate vs Monthly FTE Hours Saved")
plt.xlabel("Automation Rate (%)")
plt.ylabel("FTE Hours Saved Monthly")
plt.tight_layout()
plt.show()

## 6. Key Business Metrics

In [ ]:
# Overall KPI summary
total_spend = df["AI_Spend_USD_Million"].sum()
total_savings = df["Estimated_Cost_Savings_USD_Million"].sum()
total_projects = df["AI_Projects"].sum()
avg_roi = df["ROI_Multiple"].mean()
avg_automation = df["Automation_Rate_Percent"].mean()
total_fte_hours = df["FTE_Hours_Saved_Monthly"].sum()
net_benefit = df["Net_Benefit_USD_Million"].sum()

kpi_summary = pd.DataFrame({
    "Metric": [
        "Total AI Spend (USD Million)",
        "Total Estimated Cost Savings (USD Million)",
        "Net Benefit (USD Million)",
        "Total AI Projects",
        "Average ROI Multiple",
        "Average Automation Rate (%)",
        "Total Monthly FTE Hours Saved"
    ],
    "Value": [
        total_spend,
        total_savings,
        net_benefit,
        total_projects,
        avg_roi,
        avg_automation,
        total_fte_hours
    ]
})

display(kpi_summary)

## 7. Top and Bottom Performance

In [ ]:
# Highest ROI records
top_roi = df.nlargest(10, "ROI_Multiple")[[
    "Year", "Country", "Bank", "AI_Use_Case", "Technology",
    "AI_Spend_USD_Million", "Estimated_Cost_Savings_USD_Million",
    "ROI_Multiple", "Maturity"
]]

print("Top 10 ROI records:")
display(top_roi)

# Highest automation records
top_automation = df.nlargest(10, "Automation_Rate_Percent")[[
    "Year", "Country", "Bank", "AI_Use_Case", "Automation_Rate_Percent",
    "FTE_Hours_Saved_Monthly", "Maturity"
]]

print("Top 10 automation records:")
display(top_automation)

## 8. Automated Business Insights

In [ ]:
# Generate concise data-driven insights

highest_spend_country = country_spend.iloc[0]
highest_spend_region = region_spend.iloc[0]
highest_roi_tech = technology_roi.iloc[0]
highest_automation_maturity = maturity_automation.iloc[0]
highest_use_case = use_case_projects.iloc[0]

print("BUSINESS INSIGHTS")
print("-" * 60)

print(
    f"1. Highest total AI spending country: "
    f"{highest_spend_country['Country']} "
    f"({highest_spend_country['AI_Spend_USD_Million']:.2f} USD million)."
)

print(
    f"2. Highest total AI spending region: "
    f"{highest_spend_region['Region']} "
    f"({highest_spend_region['AI_Spend_USD_Million']:.2f} USD million)."
)

print(
    f"3. Technology with highest average ROI: "
    f"{highest_roi_tech['Technology']} "
    f"({highest_roi_tech['ROI_Multiple']:.2f}x average ROI)."
)

print(
    f"4. Maturity category with highest average automation rate: "
    f"{highest_automation_maturity['Maturity']} "
    f"({highest_automation_maturity['Automation_Rate_Percent']:.2f}%)."
)

print(
    f"5. Use case with the highest number of AI projects: "
    f"{highest_use_case['AI_Use_Case']} "
    f"({highest_use_case['AI_Projects']:.0f} projects)."
)

print(
    f"6. Overall AI spending: {total_spend:.2f} USD million, "
    f"while estimated cost savings are {total_savings:.2f} USD million."
)

print(
    f"7. Average ROI across all records is {avg_roi:.2f}x "
    f"and average automation rate is {avg_automation:.2f}%."
)

## 9. Save Cleaned Dataset

The cleaned and feature-engineered dataset can be used later in SQL and Power BI.

In [ ]:
# Save processed dataset
OUTPUT_FILE = "AI_Banking_Investment_Analysis_Cleaned.csv"
df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved as: {OUTPUT_FILE}")
print("Final dataset shape:", df.shape)

## 10. Final Conclusion

This analysis examines AI investment and business impact across banking organizations.

The notebook covers:
- Pandas for data loading, cleaning, grouping, aggregation and analysis.
- NumPy for numerical calculations and derived metrics.
- Matplotlib and Seaborn for all charts, plots and the correlation heatmap.
- Investment, cost savings, ROI, automation, FTE hours saved and AI project analysis.
- Country, region, bank, technology, maturity and use-case comparisons.

### Resume-ready project description
**AI Investment & Business Impact Analysis:** Performed end-to-end data cleaning and exploratory data analysis on banking AI investment data using Python, Pandas and NumPy. Analyzed AI spending, cost savings, ROI, automation and project adoption across countries, regions, technologies and use cases. Built business-focused visualizations and correlation heatmaps using Matplotlib and Seaborn and generated actionable insights for investment and operational impact.